# Entrenamiento y evaluación en Colab

Notebook principal nuevo del proyecto.

In [ ]:
from src.defaults import get_default_config, summarize_config
from src.augmentations import (
    get_supervised_train_augmentation,
    get_weak_augmentation,
    get_strong_augmentation,
)
from src.preprocessing import preprocess_image_and_mask

cfg = get_default_config()
cfg["image_preproc"] = "base"
cfg["mask_smoothing"] = "none"
cfg["aug_gaussian_noise_p"] = 0.30
cfg["aug_clahe_p"] = 0.10

print(summarize_config(cfg))

In [ ]:
prep = preprocess_image_and_mask(
    image_uint8=image_np,
    mask_uint8=mask_np,
    target_size=cfg["target_size"],
    use_pad=cfg["use_pad"],
    imagenet_norm=cfg["imagenet_norm"],
    image_preproc=cfg["image_preproc"],
    mask_smoothing=cfg["mask_smoothing"],
    debug=cfg["debug"],
)

In [ ]:
from src.defaults import get_default_config, summarize_config
from src.augmentations import get_supervised_train_augmentation, get_weak_augmentation, get_strong_augmentation
from src.datasets import build_supervised_datasets, build_unlabeled_datasets, build_dataloaders
from src.train import run_training
from src.evaluate import evaluate_checkpoint
from src.visualization import show_config_summary, show_dataset_examples

cfg = get_default_config()

cfg["img_root"] = "/content/drive/MyDrive/TU_DATASET/images"
cfg["msk_root"] = "/content/drive/MyDrive/TU_DATASET/masks"
cfg["exp_dir"] = "/content/drive/MyDrive/tesis_runs/exp_001"

show_config_summary(cfg)

train_tf = get_supervised_train_augmentation(cfg)
weak_tf = get_weak_augmentation(cfg)
strong_tf = get_strong_augmentation(cfg)

train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(cfg, weak_tf=weak_tf, strong_tf=strong_tf)

show_dataset_examples(train_ds, n=3)

loaders = build_dataloaders(
    cfg,
    train_ds=train_ds,
    val_ds=val_ds,
    test_ds=test_ds,
    unlabeled_ds=unlabeled_ds,
    temporal_unlab_ds=temporal_unlab_ds,
)

artifacts = run_training(cfg, loaders)
results = evaluate_checkpoint(cfg, artifacts["model"], loaders, artifacts["best_path"], artifacts["history"])